# Galaxy Music — Colab Poller

Run this while you want on-demand downloads to work. It's safe to run in multiple tabs/browsers at once — each instance atomically claims a request before working on it, so nothing gets downloaded twice.

**Steps:** Run the two cells below in order. The second cell asks for your Supabase URL and service_role key (input is hidden), then runs until you stop it (Runtime → Interrupt execution) or Colab disconnects the session.

In [ ]:
!pip -q install yt-dlp supabase requests ytmusicapi
!apt -y -qq install ffmpeg > /dev/null

In [ ]:
# Upload cookies.txt (exported from your browser's logged-in YouTube session).
# This is what lets downloads work from Colab's servers -- without it, YouTube
# blocks the request as a bot, which is the error you were hitting.
# See earlier in this chat for how to export cookies.txt with a browser extension.
from google.colab import files
import shutil, os

WORKDIR = "/content/tmp"
os.makedirs(WORKDIR, exist_ok=True)
COOKIES_PATH = os.path.join(WORKDIR, "cookies.txt")

print("Upload your cookies.txt file:")
uploaded = files.upload()
uploaded_name = list(uploaded.keys())[0]
shutil.move(uploaded_name, COOKIES_PATH)
print(f"Saved to {COOKIES_PATH}")

In [ ]:
import os, re, time, traceback
from getpass import getpass
import yt_dlp
import requests as http
from supabase import create_client
from ytmusicapi import YTMusic

SUPABASE_URL = "https://dlxbwrrodavcyppsoysk.supabase.co"
try:
    from google.colab import userdata
    SUPABASE_SERVICE_KEY = userdata.get("SUPABASE_SERVICE_KEY")
except Exception:
    SUPABASE_SERVICE_KEY = getpass("Paste your Supabase service_role key here, then press Enter: ")
POLL_INTERVAL = 15


sb = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)
ytmusic = YTMusic()


def claim(table, row_id):
    resp = sb.table(table).update({"status": "processing"}).eq("id", row_id).eq("status", "pending").execute()
    return len(resp.data) > 0


def process_request(req):
    request_id = req["id"]
    youtube_id = (req.get("youtube_id") or "").strip()
    query = (req.get("query") or "").strip()
    passed_title = (req.get("title") or "").strip()
    passed_artist = (req.get("artist") or "").strip()
    print(f"\n--- Processing request {request_id}: {query or youtube_id} ---")

    try:
        target = f"https://www.youtube.com/watch?v={youtube_id}" if youtube_id else f"ytsearch1:{query}"
        ydl_opts = {
            "format": "bestaudio/best",
            "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "mp3", "preferredquality": "192"}],
            "outtmpl": os.path.join(WORKDIR, "%(id)s.%(ext)s"),
            "quiet": False,
            "noplaylist": True,
        "extractor_args": {"youtube": {"player_client": ["android", "web"]}},
        "cookiefile": COOKIES_PATH,
        }
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(target, download=True)
            entry = info["entries"][0] if "entries" in info else info

        video_id = entry["id"]
        title = passed_title or re.sub(r"\(Official.*?\)|\[Lyrics\]|\(Lyrics\)", "", entry.get("title", "Unknown")).strip()
        artist = passed_artist or entry.get("uploader", "Unknown Artist")
        mp3_path = os.path.join(WORKDIR, f"{video_id}.mp3")
        if not os.path.exists(mp3_path):
            raise RuntimeError("mp3 not produced by yt-dlp")

        base_filename = video_id
        with open(mp3_path, "rb") as f:
            sb.storage.from_("songs").upload(f"{base_filename}.mp3", f, {"content-type": "audio/mpeg"})
        audio_url = sb.storage.from_("songs").get_public_url(f"{base_filename}.mp3")

        cover_url = None
        thumb_url = entry.get("thumbnail")
        if thumb_url:
            try:
                img_bytes = http.get(thumb_url, timeout=10).content
                sb.storage.from_("covers").upload(f"{base_filename}.jpg", img_bytes, {"content-type": "image/jpeg"})
                cover_url = sb.storage.from_("covers").get_public_url(f"{base_filename}.jpg")
            except Exception as e:
                print(f"Thumbnail skipped: {e}")

        song_insert = sb.table("songs").insert({
            "filename": f"{base_filename}.mp3",
            "title": title,
            "artist": artist,
            "youtube_id": video_id,
            "storage_path": audio_url,
            "cover_path": cover_url,
        }).execute()
        new_song_id = song_insert.data[0]["id"]

        playlist_id = req.get("playlist_id")
        if playlist_id:
            sb.table("playlist_songs").insert({"playlist_id": playlist_id, "song_id": new_song_id}).execute()

        sb.table("requests").update({"status": "done"}).eq("id", request_id).execute()
        print(f"Done: {artist} - {title}")
        if os.path.exists(mp3_path):
            os.remove(mp3_path)

    except Exception as e:
        traceback.print_exc()
        sb.table("requests").update({"status": "failed", "error": str(e)}).eq("id", request_id).execute()


def process_mix(mix):
    mix_id = mix["id"]
    print(f"\n--- Processing mix {mix_id} ---")
    try:
        total_added = mix.get("total_added") or 0
        max_total = mix.get("max_total") or 30
        batch_size = mix.get("track_limit") or 5
        remaining = max_total - total_added
        if remaining <= 0:
            sb.table("mixes").update({"status": "done", "active": False}).eq("id", mix_id).execute()
            return
        batch_size = min(batch_size, remaining)

        seed_video_id = mix.get("last_video_id")
        if not seed_video_id:
            seed_song = sb.table("songs").select("*").eq("id", mix["seed_song_id"]).single().execute().data
            seed_video_id = seed_song["youtube_id"]

        watch_playlist = ytmusic.get_watch_playlist(videoId=seed_video_id, limit=batch_size + 1)
        tracks = watch_playlist.get("tracks", [])
        if tracks and tracks[0].get("videoId") == seed_video_id:
            tracks = tracks[1:]
        tracks = tracks[:batch_size]
        if not tracks:
            sb.table("mixes").update({"status": "done", "active": False}).eq("id", mix_id).execute()
            return

        last_video_id = seed_video_id
        for t in tracks:
            artist_name = t.get("artists", [{}])[0].get("name", "") if t.get("artists") else ""
            query = f"{artist_name} {t.get('title', '')}".strip()
            sb.table("requests").insert({
                "query": query, "status": "pending", "source": "mix",
                "mix_id": mix_id, "playlist_id": mix.get("playlist_id"),
            }).execute()
            if t.get("videoId"):
                last_video_id = t["videoId"]

        new_total = total_added + len(tracks)
        sb.table("mixes").update({
            "status": "done", "last_video_id": last_video_id,
            "total_added": new_total, "active": new_total < max_total,
        }).eq("id", mix_id).execute()
        print(f"Queued {len(tracks)} more tracks for mix {mix_id} ({new_total}/{max_total} total).")

    except Exception as e:
        traceback.print_exc()
        sb.table("mixes").update({"status": "failed"}).eq("id", mix_id).execute()


print(f"Polling every {POLL_INTERVAL}s. Runtime -> Interrupt execution to stop.")
while True:
    try:
        pending_requests = sb.table("requests").select("*").eq("status", "pending").execute().data
        for req in pending_requests:
            if claim("requests", req["id"]):
                process_request(req)

        pending_mixes = sb.table("mixes").select("*").eq("status", "pending").execute().data
        for mix in pending_mixes:
            if claim("mixes", mix["id"]):
                process_mix(mix)
    except Exception as e:
        print(f"Poll loop error (will retry): {e}")
    time.sleep(POLL_INTERVAL)